In [49]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Sequential
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
from PIL import Image, UnidentifiedImageError
import tensorflow as tf

In [50]:
SALE_PATH = "../data/interim/apt/apt_with_long_lat.csv"
IMAGES_DIR = '../data/interim/satellites/rot_90'

In [51]:
sale = pd.read_csv(SALE_PATH)

In [52]:
sale.dropna(inplace=True)

In [53]:
sale.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23395 entries, 0 to 23394
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   단지명         23395 non-null  object 
 1   전용면적(㎡)     23395 non-null  float64
 2   층           23395 non-null  int64  
 3   건축년도        23395 non-null  int64  
 4   도로명         23395 non-null  object 
 5   면적당 단가(만원)  23395 non-null  float64
 6   아파트 나이      23395 non-null  int64  
 7   계약일자        23395 non-null  object 
 8   alpha       23395 non-null  float64
 9   경도          23395 non-null  float64
 10  위도          23395 non-null  float64
dtypes: float64(5), int64(3), object(3)
memory usage: 2.0+ MB


In [54]:
sale.isna().sum()

단지명           0
전용면적(㎡)       0
층             0
건축년도          0
도로명           0
면적당 단가(만원)    0
아파트 나이        0
계약일자          0
alpha         0
경도            0
위도            0
dtype: int64

In [55]:
# ===== 이미지 피처 추출 (OpenCV 사용) ===== #
valid_exts = ('.jpg', '.jpeg', '.png')
image_paths = [
    os.path.join(IMAGES_DIR, fname)
    for fname in sorted(os.listdir(IMAGES_DIR))
    if os.path.isfile(os.path.join(IMAGES_DIR, fname)) and fname.lower().endswith(valid_exts)
]
len(image_paths)

23395

In [56]:
def extract_image_features(image_paths, resnet_model):
    features = []
    for path in tqdm(image_paths):
        try:
            # OpenCV로 이미지 로드
            import cv2
            img = cv2.imread(path)
            if img is None:
                print(f"⚠️ 이미지 읽기 실패: {path}")
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # RGB 변환
            img = cv2.resize(img, (224, 224))  # ResNet50 입력 크기

            # 배열 변환 및 전처리
            x = np.expand_dims(img, axis=0)
            x = tf.keras.applications.resnet50.preprocess_input(x)

            # 피처 추출
            feat = resnet_model.predict(x, verbose=0)
            features.append(feat.flatten())
        except Exception as e:
            print(f"❌ 예기치 않은 에러: {path} - {e}")
    return np.array(features)

In [57]:
# ResNet50 모델

base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(224,224,3))
fc = Dense(2048, activation=None, trainable=False)(base_model.output)
resnet_model = Model(inputs=base_model.input, outputs=fc)

# 피처 추출
image_features = extract_image_features(image_paths, resnet_model)
df_image_features = pd.DataFrame(image_features, columns=[f'feature_{i}' for i in range(image_features.shape[1])])
df_price = sale[['면적당 단가(만원)']].reset_index(drop=True)

  0%|          | 0/23395 [00:00<?, ?it/s]

2025-09-07 17:25:33.124593: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


In [58]:
df_image_features.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,-2.845212,0.695857,-0.422789,0.464826,-1.566503,0.289307,-0.010609,1.549660,1.682617,-1.311973,...,0.560369,-1.173868,-0.531488,1.971534,-1.937145,-0.450683,-0.611337,-1.345169,1.100428,1.825825
1,-1.700194,0.626772,0.872182,-0.392932,-2.504396,0.117693,1.232154,1.191747,0.566030,-1.937688,...,0.585697,-1.808958,-1.109046,2.075178,-1.293195,0.075647,-0.779950,-1.596188,1.879856,0.659762
2,-0.903260,0.644358,0.601003,0.236581,-3.235497,-1.449778,0.995526,0.682063,0.786443,-3.068380,...,1.162668,-0.300450,-0.726406,1.230919,-1.061582,0.541388,0.356319,-1.540418,1.611366,1.068733
3,-0.874816,-0.340900,-0.525640,-0.046562,-1.138974,1.002604,0.549764,-0.007571,1.047784,-0.820854,...,1.571008,-0.593050,0.057871,0.602005,0.867684,0.033139,-0.824084,-0.880277,0.605881,1.618342
4,-1.570662,1.157273,-0.649633,0.520850,-2.046717,-0.344650,0.256032,1.493597,0.657867,-0.873588,...,0.944249,-1.922096,-1.521093,1.681462,-1.544008,-0.016917,-1.173358,-0.989139,0.605088,2.208839


In [59]:
len(df_image_features)

23395

In [70]:
df_image_features.to_csv("../data/interim/90feature.csv",index=False)

In [60]:
df_image_features.isna().sum()

feature_0       0
feature_1       0
feature_2       0
feature_3       0
feature_4       0
               ..
feature_2043    0
feature_2044    0
feature_2045    0
feature_2046    0
feature_2047    0
Length: 2048, dtype: int64

In [61]:
sale.head()

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자,alpha,경도,위도
0,중앙하이츠,59.91,10,1998,덕릉로84길 7,6.766156,22,2020-07-11,0.266667,127.076754,37.659925
1,북아현경남,59.77,7,1996,북아현로 40,7.499383,24,2020-07-11,1.000000,126.956170,37.561195
2,강남엘에이치1단지,84.83,6,2013,헌릉로571길 20,7.401580,7,2020-07-11,1.000000,127.102102,37.467098
3,중계센트럴파크,59.75,13,2016,덕릉로70가길 21,7.066081,4,2020-07-11,1.000000,127.059507,37.642869
4,주공17단지,49.94,7,1989,덕릉로66길 17,6.967225,31,2020-07-11,0.000000,127.053952,37.643712


In [62]:
len(sale)

23395

In [63]:
sale.isna().sum()

단지명           0
전용면적(㎡)       0
층             0
건축년도          0
도로명           0
면적당 단가(만원)    0
아파트 나이        0
계약일자          0
alpha         0
경도            0
위도            0
dtype: int64

In [64]:
df_combined = pd.concat([sale, df_image_features], axis=1)

In [65]:
df_combined.head()

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자,alpha,경도,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,중앙하이츠,59.91,10,1998,덕릉로84길 7,6.766156,22,2020-07-11,0.266667,127.076754,...,0.560369,-1.173868,-0.531488,1.971534,-1.937145,-0.450683,-0.611337,-1.345169,1.100428,1.825825
1,북아현경남,59.77,7,1996,북아현로 40,7.499383,24,2020-07-11,1.000000,126.956170,...,0.585697,-1.808958,-1.109046,2.075178,-1.293195,0.075647,-0.779950,-1.596188,1.879856,0.659762
2,강남엘에이치1단지,84.83,6,2013,헌릉로571길 20,7.401580,7,2020-07-11,1.000000,127.102102,...,1.162668,-0.300450,-0.726406,1.230919,-1.061582,0.541388,0.356319,-1.540418,1.611366,1.068733
3,중계센트럴파크,59.75,13,2016,덕릉로70가길 21,7.066081,4,2020-07-11,1.000000,127.059507,...,1.571008,-0.593050,0.057871,0.602005,0.867684,0.033139,-0.824084,-0.880277,0.605881,1.618342
4,주공17단지,49.94,7,1989,덕릉로66길 17,6.967225,31,2020-07-11,0.000000,127.053952,...,0.944249,-1.922096,-1.521093,1.681462,-1.544008,-0.016917,-1.173358,-0.989139,0.605088,2.208839


In [66]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23395 entries, 0 to 23394
Columns: 2059 entries, 단지명 to feature_2047
dtypes: float32(2048), float64(5), int64(3), object(3)
memory usage: 184.7+ MB


In [67]:
df_combined.isna().sum()

단지명             0
전용면적(㎡)         0
층               0
건축년도            0
도로명             0
               ..
feature_2043    0
feature_2044    0
feature_2045    0
feature_2046    0
feature_2047    0
Length: 2059, dtype: int64

In [68]:
df_combined.dropna(inplace=True)

In [69]:
df_combined.to_csv("../data/interim/resnet90deg.csv", index=False)

# 간단하게 MLP 돌려보기

In [92]:
df_combined.head()

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자,alpha,경도,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,중앙하이츠,59.91,10.0,1998.0,덕릉로84길 7,6.766156,22.0,2020-07-11,0.266667,127.076754,...,2.004844,-2.512005,-1.100615,-0.027693,-1.204624,2.213226,1.975165,0.900340,0.369453,0.569278
1,북아현경남,59.77,7.0,1996.0,북아현로 40,7.499383,24.0,2020-07-11,1.000000,126.956170,...,2.295805,-2.131941,-1.488998,-0.062864,-0.122249,2.217655,1.203446,1.180406,0.273386,0.130200
2,강남엘에이치1단지,84.83,6.0,2013.0,헌릉로571길 20,7.401580,7.0,2020-07-11,1.000000,127.102102,...,1.964717,-2.031952,-1.270553,-0.651409,-1.210025,1.739807,1.211842,1.265154,0.637330,-0.447079
3,중계센트럴파크,59.75,13.0,2016.0,덕릉로70가길 21,7.066081,4.0,2020-07-11,1.000000,127.059507,...,1.513175,-1.890251,-0.957274,-0.138104,-0.749877,1.704004,1.395738,0.735970,0.548623,-0.146055
4,주공17단지,49.94,7.0,1989.0,덕릉로66길 17,6.967225,31.0,2020-07-11,0.000000,127.053952,...,2.188202,-2.315390,-1.987213,0.255648,0.739055,2.142885,1.633339,1.013102,0.846412,0.591645


In [119]:
df = df_combined

In [120]:
sale.columns

Index(['단지명', '전용면적(㎡)', '층', '건축년도', '도로명', '면적당 단가(만원)', '아파트 나이', '계약일자',
       'alpha', '경도', '위도'],
      dtype='object')

In [121]:
df.columns

Index(['전용면적(㎡)', '층', '건축년도', '면적당 단가(만원)', '아파트 나이', 'alpha', 'feature_0',
       'feature_1', 'feature_2', 'feature_3',
       ...
       'feature_2038', 'feature_2039', 'feature_2040', 'feature_2041',
       'feature_2042', 'feature_2043', 'feature_2044', 'feature_2045',
       'feature_2046', 'feature_2047'],
      dtype='object', length=2054)

In [122]:
df.drop(['단지명','도로명','계약일자','경도','위도'], axis=1, inplace=True)

KeyError: "['단지명', '도로명', '계약일자', '경도', '위도'] not found in axis"

In [123]:
df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,59.91,10.0,1998.0,6.766156,22.0,0.266667,1.840863,0.455314,1.392201,-1.213092,...,2.004844,-2.512005,-1.100615,-0.027693,-1.204624,2.213226,1.975165,0.900340,0.369453,0.569278
1,59.77,7.0,1996.0,7.499383,24.0,1.000000,0.512528,0.327763,0.756215,-2.196653,...,2.295805,-2.131941,-1.488998,-0.062864,-0.122249,2.217655,1.203446,1.180406,0.273386,0.130200
2,84.83,6.0,2013.0,7.401580,7.0,1.000000,0.274194,-0.137119,1.368243,-1.201620,...,1.964717,-2.031952,-1.270553,-0.651409,-1.210025,1.739807,1.211842,1.265154,0.637330,-0.447079
3,59.75,13.0,2016.0,7.066081,4.0,1.000000,0.402139,0.171405,0.707894,-0.902115,...,1.513175,-1.890251,-0.957274,-0.138104,-0.749877,1.704004,1.395738,0.735970,0.548623,-0.146055
4,49.94,7.0,1989.0,6.967225,31.0,0.000000,1.091872,0.706208,0.801780,-1.592087,...,2.188202,-2.315390,-1.987213,0.255648,0.739055,2.142885,1.633339,1.013102,0.846412,0.591645


In [124]:

# 수치형 컬럼만 선택
num_cols = df.select_dtypes(include=['int64', 'float64', 'float32']).columns

scaler = MinMaxScaler()
scaled_array = scaler.fit_transform(df[num_cols])

# DataFrame으로 변환, 기존 인덱스 유지
df_scaled = pd.DataFrame(scaled_array, columns=num_cols, index=df.index)

# 필요 시 기존 비수치형 컬럼과 합치기
non_num_cols = df.select_dtypes(exclude=['int64', 'float64', 'float32'])
df_final = pd.concat([df_scaled, non_num_cols], axis=1)

In [126]:
df_final.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,feature_0,feature_1,feature_2,feature_3,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,0.160252,0.185714,0.578125,0.437752,0.377049,0.266667,0.636647,0.366434,0.545539,0.498299,...,0.606152,0.366731,0.428327,0.458830,0.378163,0.456591,0.603196,0.626604,0.558743,0.715487
1,0.159795,0.142857,0.546875,0.615627,0.409836,1.000000,0.366485,0.342304,0.382654,0.291086,...,0.671792,0.444659,0.337388,0.450494,0.598046,0.457651,0.417910,0.696872,0.540540,0.619273
2,0.241536,0.128571,0.812500,0.591901,0.131148,1.000000,0.318012,0.254358,0.539403,0.500716,...,0.597100,0.465161,0.388537,0.310996,0.377066,0.343248,0.419925,0.718135,0.609502,0.492775
3,0.159730,0.228571,0.859375,0.510512,0.081967,1.000000,0.344034,0.312724,0.370278,0.563815,...,0.495234,0.494215,0.461891,0.432661,0.470545,0.334677,0.464078,0.585364,0.592693,0.558738
4,0.127732,0.142857,0.437500,0.486530,0.524590,0.000000,0.484314,0.413897,0.394324,0.418454,...,0.647517,0.407045,0.220731,0.525989,0.773019,0.439750,0.521125,0.654896,0.649120,0.720389


In [129]:

# 1. 입력과 타깃 분리
X = df_final.drop(columns=['면적당 단가(만원)']).values  # 예: 타깃 제외
y = df_final['면적당 단가(만원)'].values.reshape(-1, 1)

# 2. MLP 모델 정의
model = Sequential([
    Dense(512, activation='relu', input_shape=(X.shape[1],)),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1)  # 회귀용 출력
])

# 3. 모델 컴파일
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 4. 학습
model.fit(X, y, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/50
581/581 [==============================] - 1s 2ms/step - loss: 0.1248 - mae: 0.1385 - val_loss: 0.0220 - val_mae: 0.1147
Epoch 2/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0165 - mae: 0.1020 - val_loss: 0.0173 - val_mae: 0.1054
Epoch 3/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0157 - mae: 0.0993 - val_loss: 0.0173 - val_mae: 0.1043
Epoch 4/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0148 - mae: 0.0964 - val_loss: 0.0177 - val_mae: 0.1046
Epoch 5/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0146 - mae: 0.0958 - val_loss: 0.0169 - val_mae: 0.1042
Epoch 6/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0145 - mae: 0.0953 - val_loss: 0.0172 - val_mae: 0.1037
Epoch 7/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0144 - mae: 0.0948 - val_loss: 0.0166 - val_mae: 0.1025
Epoch 8/50
581/581 [==============================] - 1s 2ms/step - loss: 0.

In [134]:
# 8:1:1 비율로 분할
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# MLP 모델 학습
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=50,
                    batch_size=32)

# 예측
y_pred = model.predict(X_test)

Epoch 1/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0153 - mae: 0.0976 - val_loss: 0.0144 - val_mae: 0.0954
Epoch 2/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0150 - mae: 0.0970 - val_loss: 0.0143 - val_mae: 0.0949
Epoch 3/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0150 - mae: 0.0970 - val_loss: 0.0144 - val_mae: 0.0952
Epoch 4/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0150 - mae: 0.0970 - val_loss: 0.0143 - val_mae: 0.0950
Epoch 5/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0150 - mae: 0.0970 - val_loss: 0.0144 - val_mae: 0.0952
Epoch 6/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0150 - mae: 0.0969 - val_loss: 0.0143 - val_mae: 0.0951
Epoch 7/50
581/581 [==============================] - 1s 2ms/step - loss: 0.0150 - mae: 0.0969 - val_loss: 0.0144 - val_mae: 0.0949
Epoch 8/50
581/581 [==============================] - 1s 2ms/step - loss: 0.

In [136]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np

kf = KFold(n_splits=10, shuffle=True, random_state=42)
val_mae_list = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    # 스케일링
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    # 모델 정의
    model = Sequential([
        Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # 학습
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

    # 검증
    val_mae = model.evaluate(X_val, y_val, verbose=0)[1]  # mae
    val_mae_list.append(val_mae)

print("10-fold CV val_mae:", np.mean(val_mae_list), "+/-", np.std(val_mae_list))

10-fold CV val_mae: 0.10288864150643348 +/- 0.0025505216416210285


In [141]:
X

array([[0.16025181, 0.18571429, 0.578125  , ..., 0.62660405, 0.55874337,
        0.71548745],
       [0.15979516, 0.14285714, 0.546875  , ..., 0.69687231, 0.54054033,
        0.61927291],
       [0.24153565, 0.12857143, 0.8125    , ..., 0.71813546, 0.60950195,
        0.49277455],
       ...,
       [0.67052645, 0.07142857, 0.953125  , ..., 0.6345571 , 0.5988409 ,
        0.42782332],
       [0.19841477, 0.12857143, 0.625     , ..., 0.52322651, 0.42927484,
        0.72373994],
       [0.29763846, 0.15714286, 0.859375  , ..., 0.64295725, 0.59246376,
        0.74475898]])

In [145]:
sale.columns

Index(['단지명', '전용면적(㎡)', '층', '건축년도', '도로명', '면적당 단가(만원)', '아파트 나이', '계약일자',
       'alpha', '경도', '위도'],
      dtype='object')

In [146]:
sale.drop(['단지명','도로명','위도','경도','계약일자'], axis=1, inplace=True)

In [148]:
X = sale.drop(columns=['면적당 단가(만원)']).values
y = sale['면적당 단가(만원)'].values.reshape(-1, 1)

In [149]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
val_mae_list = []

In [150]:

for train_index, val_index in kf.split(X):
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    # 스케일링
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    # 모델 정의
    model = Sequential([
        Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # 학습
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

    # 검증
    val_mae = model.evaluate(X_val, y_val, verbose=0)[1]  # mae
    val_mae_list.append(val_mae)

print("10-fold CV val_mae:", np.mean(val_mae_list), "+/-", np.std(val_mae_list))

10-fold CV val_mae: 0.32622748911380767 +/- 0.010929465421123889


In [151]:
1+1

2